# 60 — Reach Probability + R/R Grid

Builds the hybrid reach-probability estimators (`azlib.rr
.reach_prob_estimator`, frozen on THIS dataset — an exploratory,
single-dataset notebook, not a train/OOS split), sweeps the full `(tgt_x,
sl_x)` grid (`azlib.rr.rr_grid`), and reports the selected levels
(`azlib.zones.select_levels_safe`). Down-side (`reach_down`) wiring uses the
SAME negated-`{tf}_low_diff_prc` convention `azlib.zones.build_rr_levels`
documents (see that module's docstring) — reproduced here (not imported
from `build_rr_levels` directly) only so the intermediate `rr_grid`
DataFrame itself can be saved/plotted; `azlib.rr`'s own functions do all the
actual math. `candle_size` reuses (DRY) `azlib.validate`'s own
median-completed-candle-range helper rather than recomputing it.

In [ ]:
tf = 15
direction = "long"
fee = 0.001

In [ ]:
import sys
from pathlib import Path

# Bootstrap: notebooks/action_zones/{azlib,nb} onto sys.path. Assumes cwd is
# the repo root (/code) -- true for the experiment Docker service
# (docker-compose.experiment.yml's working_dir: /code; papermill's own
# cwd=None default inherits the invoking process's cwd) and for the
# automated smoke test (tests/test_layer9_notebooks.py sets the papermill
# subprocess's cwd to the repo root to match). helpers.py/indicators.*
# resolve for free via the image's own PYTHONPATH=/code.
_AZ_ROOT = Path.cwd() / "notebooks" / "action_zones"
for _p in (_AZ_ROOT, _AZ_ROOT / "nb"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import nbutil

In [ ]:
import numpy as np

from azlib.loader import load_wide_df
from azlib.rr import reach_prob_estimator, rr_grid
from azlib.zones import select_levels_safe
from azlib.validate import _candle_size  # reuse (DRY), see markdown cell above

In [ ]:
wide_df = load_wide_df()
out_dir = nbutil.results_dir(tf, direction, "rr")

high_diff_prc_train = wide_df[f"{tf}_high_diff_prc"].dropna().to_numpy()
low_diff_prc_train = wide_df[f"{tf}_low_diff_prc"].dropna().to_numpy()

reach_up = reach_prob_estimator(high_diff_prc_train)
reach_down = reach_prob_estimator(-low_diff_prc_train)  # negated -- see azlib.rr's sign convention

x_grid = np.round(np.arange(0.0, 2.01, 0.25), 2)
candle_size = _candle_size(wide_df, tf)

grid = rr_grid(reach_up, reach_down, x_grid, fee=fee, candle_size=candle_size, direction=direction)
grid_path = f"{out_dir}/rr_grid.csv"
grid.to_csv(grid_path, index=False)
print(f"{len(grid)} (tgt_x, sl_x) combos -> {grid_path}")

In [ ]:
levels = select_levels_safe(grid)
levels_path = nbutil.write_json_report(f"{out_dir}/selected_levels.json", levels)
print(levels)

In [ ]:
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_grid, reach_up(x_grid), label="reach_up P(X>=level)", color="tab:green")
ax.plot(x_grid, reach_down(x_grid), label="reach_down P(X>=level)", color="tab:red")
ax.set_xlabel("level (x_grid)")
ax.set_ylabel("P(reach)")
ax.set_title(f"Reach probability sanity — tf={tf} direction={direction}")
ax.legend(loc="best")
fig.tight_layout()
reach_plot_path = f"{out_dir}/reach_prob_sanity.png"
fig.savefig(reach_plot_path)
plt.close(fig)
print(reach_plot_path)